# TREC-RAG 2024 Dataset Overview

This notebook downloads core public artifacts for TREC-RAG 2024 and summarizes:
- number of queries
- judged/relevant docs from qrels
- average relevant docs per query
- BM25 dev-run coverage
- a few query and relevance examples

Note: the full MS MARCO v2.1 segment corpus is very large (120M+ segments).
This notebook uses official topics/qrels and released BM25 run files for practical local analysis.

In [1]:
from pathlib import Path
import subprocess
import pandas as pd

DATA_DIR = Path('data/trec_rag_2024')
DATA_DIR.mkdir(parents=True, exist_ok=True)

URLS = {
    'topics.rag24.test.txt': 'https://raw.githubusercontent.com/castorini/anserini-tools/master/topics-and-qrels/topics.rag24.test.txt',
    'qrels.rag24.test.txt': 'https://raw.githubusercontent.com/castorini/anserini-tools/master/topics-and-qrels/qrels.rag24.test.txt',
    'run.bm25.rag24.raggy-dev.txt': 'https://raw.githubusercontent.com/castorini/ragnarok_data/main/rag24/retrieve_results/BM25/run.msmarco-v2.1-doc-segmented.bm25.rag24.raggy-dev.txt',
    'run.bm25.rag24.researchy-dev.txt': 'https://raw.githubusercontent.com/castorini/ragnarok_data/main/rag24/retrieve_results/BM25/run.msmarco-v2.1-doc-segmented.bm25.rag24.researchy-dev.txt',
}

def _download(url: str, target: Path) -> None:
    if target.exists() and target.stat().st_size > 0:
        print(f'[cached] {target}')
        return
    print(f'[download] {url}')
    subprocess.run(['curl', '-L', '-o', str(target), url], check=True)

for name, url in URLS.items():
    _download(url, DATA_DIR / name)

print('Done. Files in', DATA_DIR)
for p in sorted(DATA_DIR.glob('*.txt')):
    print(f'  {p.name:45s} {p.stat().st_size:,} bytes')

[cached] data/trec_rag_2024/topics.rag24.test.txt
[cached] data/trec_rag_2024/qrels.rag24.test.txt
[cached] data/trec_rag_2024/run.bm25.rag24.raggy-dev.txt
[cached] data/trec_rag_2024/run.bm25.rag24.researchy-dev.txt
Done. Files in data/trec_rag_2024
  qrels.rag24.test.txt                          1,201,033 bytes
  run.bm25.rag24.raggy-dev.txt                  910,820 bytes
  run.bm25.rag24.researchy-dev.txt              4,496,144 bytes
  topics.rag24.test.txt                         19,517 bytes


In [2]:
topics = pd.read_csv(
    DATA_DIR / 'topics.rag24.test.txt',
    sep='\t',
    names=['qid', 'query'],
    dtype={'qid': str},
    keep_default_na=False,
)

qrels = pd.read_csv(
    DATA_DIR / 'qrels.rag24.test.txt',
    sep='\s+',
    names=['qid', 'Q0', 'docid', 'rel'],
    dtype={'qid': str, 'docid': str},
)

run_raggy = pd.read_csv(
    DATA_DIR / 'run.bm25.rag24.raggy-dev.txt',
    sep='\s+',
    names=['qid', 'Q0', 'docid', 'rank', 'score', 'runid'],
    dtype={'qid': str, 'docid': str},
).assign(split='raggy-dev')

run_researchy = pd.read_csv(
    DATA_DIR / 'run.bm25.rag24.researchy-dev.txt',
    sep='\s+',
    names=['qid', 'Q0', 'docid', 'rank', 'score', 'runid'],
    dtype={'qid': str, 'docid': str},
).assign(split='researchy-dev')

runs = pd.concat([run_raggy, run_researchy], ignore_index=True)

qrels_pos = qrels[qrels['rel'] > 0].copy()
qrels_pos_per_q = qrels_pos.groupby('qid')['docid'].nunique().rename('n_relevant')

In [3]:
overview = {
    'n_queries_topics': int(topics['qid'].nunique()),
    'n_qrels_rows_all': int(len(qrels)),
    'n_qrels_rows_positive': int(len(qrels_pos)),
    'n_unique_docs_in_positive_qrels': int(qrels_pos['docid'].nunique()),
    'n_queries_with_positive_qrels': int(qrels_pos['qid'].nunique()),
    'avg_relevant_docs_per_query__all_301_topics': float(qrels_pos_per_q.reindex(topics['qid']).fillna(0).mean()),
    'avg_relevant_docs_per_query__only_queries_with_positive_qrels': float(qrels_pos_per_q.mean()),
}

pd.DataFrame([overview]).T.rename(columns={0: 'value'})

,value
n_queries_topics,301.000000
n_qrels_rows_all,20429.000000
n_qrels_rows_positive,12717.000000
n_unique_docs_in_positive_qrels,12650.000000
n_queries_with_positive_qrels,86.000000
avg_relevant_docs_per_query__all_301_topics,42.249169
avg_relevant_docs_per_query__only_queries_with_positive_qrels,147.872093


In [4]:
run_stats = pd.DataFrame([
    {
        'split': 'raggy-dev',
        'n_rows': len(run_raggy),
        'n_queries': run_raggy['qid'].nunique(),
        'avg_docs_per_query': run_raggy.groupby('qid')['docid'].nunique().mean(),
        'unique_docs': run_raggy['docid'].nunique(),
    },
    {
        'split': 'researchy-dev',
        'n_rows': len(run_researchy),
        'n_queries': run_researchy['qid'].nunique(),
        'avg_docs_per_query': run_researchy.groupby('qid')['docid'].nunique().mean(),
        'unique_docs': run_researchy['docid'].nunique(),
    },
    {
        'split': 'combined',
        'n_rows': len(runs),
        'n_queries': runs['qid'].nunique(),
        'avg_docs_per_query': runs.groupby('qid')['docid'].nunique().mean(),
        'unique_docs': runs['docid'].nunique(),
    },
])
run_stats

,split,n_rows,n_queries,avg_docs_per_query,unique_docs
0,raggy-dev,12000,120,100.0,11991
1,researchy-dev,60000,600,100.0,59336
2,combined,72000,720,100.0,71314


In [5]:
# Query examples
topics.sample(8, random_state=42).sort_values('qid').reset_index(drop=True)

,qid,query
0,2024-141453,what happens if evidence is lost?
1,2024-149261,what parts of culture did germans still do whe...
2,2024-158743,what was happening in germany and netherlands ...
3,2024-215090,why did america want to overthrow the hawaiian...
4,2024-219698,why human imagination important for anthropolo...
5,2024-225531,why would parents object to the teaching of ev...
6,2024-23862,do states mismanage transportation funds?
7,2024-41198,how does edi impact nursing


In [6]:
# Relevance examples: top 3 positively judged docs for a few random queries
example_qids = qrels_pos['qid'].drop_duplicates().sample(5, random_state=0).tolist()
example_qrels = (
    qrels_pos[qrels_pos['qid'].isin(example_qids)]
    .sort_values(['qid', 'rel'], ascending=[True, False])
    .groupby('qid', as_index=False)
    .head(3)
    .merge(topics, on='qid', how='left')
    [['qid', 'query', 'docid', 'rel']]
    .reset_index(drop=True)
)
example_qrels

,qid,query,docid,rel
0,2024-127266,what are some key challenges related to the re...,msmarco_v2.1_doc_25_614830727#4_1217823507,3
1,2024-127266,what are some key challenges related to the re...,msmarco_v2.1_doc_31_1674443838#0_3431968517,3
2,2024-127266,what are some key challenges related to the re...,msmarco_v2.1_doc_39_1205713220#2_2457418789,3
3,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_04_1043879213#10_2155808690,3
4,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_04_1043879213#11_2155809953,3
5,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_04_251223477#2_650543692,3
6,2024-220058,why is being born premature a risk factor for ...,msmarco_v2.1_doc_00_756157388#24_1411992163,3
7,2024-220058,why is being born premature a risk factor for ...,msmarco_v2.1_doc_31_680964661#27_1341245606,3
8,2024-220058,why is being born premature a risk factor for ...,msmarco_v2.1_doc_36_214763944#10_450925364,3
9,2024-27366,does minimalism comes from creativity?,msmarco_v2.1_doc_13_1091317306#2_2423168867,2


## Notes

- `topics.rag24.test.txt` contains 301 official test topics.
- `qrels.rag24.test.txt` contains judged relevance labels (sparse and pooled; not every topic has positive labels).
- BM25 run files are provided for the two dev sets (`raggy-dev`, `researchy-dev`) with top-100 docs per query.
- Full corpus-level segment text statistics require downloading the MS MARCO v2.1 segmented corpus (very large).

## Ground Truth LLM Response Availability

This benchmark release provides:
- topics (queries),
- qrels (document relevance judgments),
- retrieval run files.

It does **not** provide a single canonical “ground-truth LLM response text” per query in these downloaded files.

For generation evaluation, the benchmark uses relevance judgments and response-level eval protocols rather than a single gold answer string.

In [7]:
# Quick machine-check for "ground-truth LLM response text" artifacts
candidate_patterns = ['*answer*', '*answers*', '*response*', '*responses*', '*reference*', '*gold*']
candidate_files = sorted({p.name for pattern in candidate_patterns for p in DATA_DIR.glob(pattern)})

summary = {
    'n_topics': int(topics['qid'].nunique()),
    'n_positive_qrels': int(len(qrels_pos)),
    'n_queries_with_positive_qrels': int(qrels_pos['qid'].nunique()),
    'avg_rel_docs_per_query_all_topics': float(qrels_pos_per_q.reindex(topics['qid']).fillna(0).mean()),
    'avg_rel_docs_per_query_positive_only': float(qrels_pos_per_q.mean()),
    'candidate_ground_truth_response_files_found': candidate_files,
}

pd.Series(summary)

n_topics                                              301
n_positive_qrels                                    12717
n_queries_with_positive_qrels                          86
avg_rel_docs_per_query_all_topics               42.249169
avg_rel_docs_per_query_positive_only           147.872093
candidate_ground_truth_response_files_found            []
dtype: object

In [3]:
# Compact examples: a few queries and their positively judged docids
sample_qids = qrels_pos['qid'].drop_duplicates().sample(5, random_state=7).tolist()
examples_compact = (
    qrels_pos[qrels_pos['qid'].isin(sample_qids)]
    .sort_values(['qid', 'rel', 'docid'], ascending=[True, False, True])
    .groupby('qid', as_index=False)
    .head(5)
    .merge(topics, on='qid', how='left')
    [['qid', 'query', 'docid', 'rel']]
    .reset_index(drop=True)
)

examples_compact

,qid,query,docid,rel
0,2024-145979,what is vicarious trauma and how can it be cop...,msmarco_v2.1_doc_01_1812201916#16_2674214039,3
1,2024-145979,what is vicarious trauma and how can it be cop...,msmarco_v2.1_doc_01_1812201916#17_2674215398,3
2,2024-145979,what is vicarious trauma and how can it be cop...,msmarco_v2.1_doc_01_523681915#0_449763684,3
3,2024-145979,what is vicarious trauma and how can it be cop...,msmarco_v2.1_doc_01_523681915#12_449783870,3
4,2024-145979,what is vicarious trauma and how can it be cop...,msmarco_v2.1_doc_01_523681915#13_449785756,3
5,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_04_1043879213#10_2155808690,3
6,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_04_1043879213#11_2155809953,3
7,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_04_251223477#2_650543692,3
8,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_04_251223477#3_650545664,3
9,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_11_587847680#18_1059506724,3


In [12]:
# Show actual text for a few relevant docs (not just docids)
# Resolves TREC-RAG qrels docids to Pyserini msmarco-v2.1-doc-segmented index docids.

import os
import re
import subprocess

if 'examples_compact' not in globals() or examples_compact.empty:
    raise RuntimeError('Run the examples cell first so examples_compact is available and non-empty.')

sample_rows = examples_compact[['qid', 'query', 'docid', 'rel']].drop_duplicates().head(8).copy()
resolved = []

# Ensure Homebrew OpenJDK is visible to this kernel process.
openjdk_home = '/opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home'
openjdk_bin = '/opt/homebrew/opt/openjdk/bin'
if os.path.isdir(openjdk_home):
    os.environ['JAVA_HOME'] = openjdk_home
if os.path.isdir(openjdk_bin) and openjdk_bin not in os.environ.get('PATH', ''):
    os.environ['PATH'] = f"{openjdk_bin}:{os.environ.get('PATH', '')}"

print('JAVA_HOME =', os.environ.get('JAVA_HOME', '<unset>'))
java_ver = subprocess.run(['java', '-version'], capture_output=True, text=True, check=False)
print('java -version return code:', java_ver.returncode)


def normalize_docid_for_pyserini(docid: str) -> str:
    # qrels: msmarco_v2.1_doc_13_1743852482#0_2099158582
    # index: msmarco_doc_13_1743852482#0
    m = re.match(r'^msmarco_v2\.1_doc_(\d+_\d+)#(\d+)_\d+$', docid)
    if m:
        return f"msmarco_doc_{m.group(1)}#{m.group(2)}"
    return docid


try:
    from pyserini.search.lucene import LuceneSearcher

    index_name = 'msmarco-v2.1-doc-segmented'
    searcher = LuceneSearcher.from_prebuilt_index(index_name)
    print(f'Using pyserini LuceneSearcher with {index_name}.')

    for row in sample_rows.itertuples(index=False):
        norm_docid = normalize_docid_for_pyserini(row.docid)

        doc = searcher.doc(norm_docid)
        if doc is None:
            # Try raw docid as fallback.
            doc = searcher.doc(row.docid)

        if doc is None:
            raw_text = f'[doc not found in index: raw={row.docid} | normalized={norm_docid}]'
        else:
            raw_text = doc.get('contents') or ''

        snippet = ' '.join(str(raw_text).split())[:500]
        resolved.append({
            'qid': row.qid,
            'query': row.query,
            'docid_qrels': row.docid,
            'docid_index': norm_docid,
            'rel': row.rel,
            'doc_text_snippet': snippet,
        })
except Exception as e:
    print(f'Pyserini lookup failed: {e}')

if not resolved:
    print('Could not resolve document text from current local setup.')
    for row in sample_rows.itertuples(index=False):
        resolved.append({
            'qid': row.qid,
            'query': row.query,
            'docid_qrels': row.docid,
            'docid_index': normalize_docid_for_pyserini(row.docid),
            'rel': row.rel,
            'doc_text_snippet': '[Text unavailable locally yet]',
        })

pd.DataFrame(resolved)

JAVA_HOME = /opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home
java -version return code: 0


lucene-inverted.msmarco-v2.1-doc-segmented.20240418.4f9675.tar.gz: 100%|██████████| 55.9G/55.9G [1:47:30<00:00, 9.31MB/s]   


Using pyserini LuceneSearcher with msmarco-v2.1-doc-segmented.


,qid,query,docid_qrels,docid_index,rel,doc_text_snippet
0,2024-145979,what is vicarious trauma and how can it be cop...,msmarco_v2.1_doc_01_1812201916#16_2674214039,msmarco_doc_01_1812201916#16,3,
1,2024-145979,what is vicarious trauma and how can it be cop...,msmarco_v2.1_doc_01_1812201916#17_2674215398,msmarco_doc_01_1812201916#17,3,
2,2024-145979,what is vicarious trauma and how can it be cop...,msmarco_v2.1_doc_01_523681915#0_449763684,msmarco_doc_01_523681915#0,3,
3,2024-145979,what is vicarious trauma and how can it be cop...,msmarco_v2.1_doc_01_523681915#12_449783870,msmarco_doc_01_523681915#12,3,
4,2024-145979,what is vicarious trauma and how can it be cop...,msmarco_v2.1_doc_01_523681915#13_449785756,msmarco_doc_01_523681915#13,3,
5,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_04_1043879213#10_2155808690,msmarco_doc_04_1043879213#10,3,
6,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_04_1043879213#11_2155809953,msmarco_doc_04_1043879213#11,3,
7,2024-149459,what percent of students are bullied because o...,msmarco_v2.1_doc_04_251223477#2_650543692,msmarco_doc_04_251223477#2,3,


In [7]:
# Probe docid mapping between qrels format and pyserini index format
from pyserini.search.lucene import LuceneSearcher

searcher = LuceneSearcher.from_prebuilt_index('msmarco-v2-doc-segmented')
probe_docid = examples_compact.iloc[0]['docid']

candidates = [probe_docid]
if probe_docid.startswith('msmarco_v2.1_doc_'):
    tail = probe_docid.split('msmarco_v2.1_doc_', 1)[1]
    candidates.append(tail)
    if '_' in tail:
        candidates.append(tail.split('_', 1)[1])
if '#' in probe_docid:
    left, right = probe_docid.split('#', 1)
    candidates.append(left)
    candidates.append(right)

print('probe_docid:', probe_docid)
print('trying candidates:')
for c in dict.fromkeys(candidates):
    hit = searcher.doc(c)
    print(' ', c, '->', 'FOUND' if hit else 'not found')

probe_docid: msmarco_v2.1_doc_01_1812201916#16_2674214039
trying candidates:
  msmarco_v2.1_doc_01_1812201916#16_2674214039 -> not found
  01_1812201916#16_2674214039 -> not found
  1812201916#16_2674214039 -> not found
  msmarco_v2.1_doc_01_1812201916 -> not found
  16_2674214039 -> not found


In [ ]:
# Inspect docid format used by the downloaded pyserini index
query_probe = topics.iloc[0]['query']
hits = searcher.search(query_probe, k=3)
print('query_probe:', query_probe)
for i, h in enumerate(hits, 1):
    print(i, h.docid, h.score)
    d = searcher.doc(h.docid)
    text = d.get('contents') if d else ''
    print('   snippet:', ' '.join(text.split())[:160])

query_probe: what is vicarious trauma and how can it be coped with?
1 msmarco_doc_13_1743852482#0 20.877599716186523


TypeError: Document.get() takes 2 positional arguments but 3 were given

In [10]:
# Discover available prebuilt pyserini indexes related to msmarco segmented corpora
from pyserini.prebuilt_index_info import TF_INDEX_INFO, IMPACT_INDEX_INFO

cands = sorted(
    [k for k in TF_INDEX_INFO.keys() if 'msmarco' in k and ('segment' in k or 'segmented' in k)] +
    [k for k in IMPACT_INDEX_INFO.keys() if 'msmarco' in k and ('segment' in k or 'segmented' in k)]
)

pd.DataFrame({'pyserini_index_name': sorted(set(cands))}).head(200)

,pyserini_index_name
0,msmarco-v1-doc-segmented
1,msmarco-v1-doc-segmented-full
2,msmarco-v1-doc-segmented-slim
3,msmarco-v1-doc-segmented.d2q-t5
4,msmarco-v1-doc-segmented.d2q-t5-docvectors
5,msmarco-v1-doc-segmented.unicoil
6,msmarco-v1-doc-segmented.unicoil-noexp
7,msmarco-v2-doc-segmented
8,msmarco-v2-doc-segmented-full
9,msmarco-v2-doc-segmented-slim


In [13]:
# Inspect available fields in the underlying Lucene document
probe_norm = normalize_docid_for_pyserini(examples_compact.iloc[0]['docid'])
probe_doc = searcher.doc(probe_norm)

print('probe normalized docid:', probe_norm)
if probe_doc is None:
    print('Document not found in index')
else:
    print('Lucene fields:', probe_doc.lucene_document.getFields())
    print('contents length:', len(probe_doc.get('contents') or ''))
    print('raw length:', len(probe_doc.raw() or ''))
    print('raw preview:', (probe_doc.raw() or '')[:400])

probe normalized docid: msmarco_doc_01_1812201916#16
Document not found in index


In [ ]:
# Reliable way to see actual document text now: show top retrieved hits with snippets
# plus optional weak matching from qrels docid -> retrieved docs by base doc id.

import re
from pyserini.search.lucene import LuceneSearcher

searcher_v21 = LuceneSearcher.from_prebuilt_index('msmarco-v2.1-doc-segmented')

query_probe = topics.iloc[0]['query']
hits = searcher_v21.search(query_probe, k=5)

rows = []
for h in hits:
    d = searcher_v21.doc(h.docid)
    raw = d.raw() if d else ''
    text = raw if raw else (d.get('contents') if d else '')
    snippet = ' '.join(str(text).split())[:450]
    rows.append({
        'query': query_probe,
        'retrieved_docid': h.docid,
        'score': float(h.score),
        'snippet': snippet,
    })

retrieved_examples = pd.DataFrame(rows)
retrieved_examples